In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# Access Google Drive Folder
import os
os.chdir("/content/gdrive/MyDrive/LLM/CBLLMMODEL")

In [ ]:
import PyPDF2
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
import re
import spacy
nlp = spacy.load("en_core_web_sm")  # Daha güçlü için: en_core_web_trf

In [ ]:
def extract_txt_from_pdf(path):
    with open(path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        text = ""
        for page in reader.pages:
            text += page.extract_text()
    return text

In [ ]:
t = extract_txt_from_pdf("Dataset/DecisionReports/Decision_2006_10_September.pdf")

In [ ]:
doc = nlp(t)

for i in doc.sents:
    print(i.text)

In [ ]:
from spellchecker import SpellChecker

spell = SpellChecker(language='en')

sentence = "This a spell ing error exampl e."
sentence = re.sub(r'\s+', ' ', sentence).strip()
words = sentence.split()
corrected_words = [spell.correction(word) for word in words]
corrected_sentence = " ".join(corrected_words)

print("Orijinal:", sentence)
print("Düzeltilmiş:", corrected_sentence)


In [ ]:
import re
from spellchecker import SpellChecker # pip install pyspellchecker

In [ ]:

def clean_text_from_text(raw_text):
    text = raw_text.replace('\n', ' ')
    match = re.search(r'[-_–—]{2,}(.*)', text)
    if match:
        text = match.group(1)  # Sadece tirelerden SONRAKİ kısmı alıyoruz

    # 3. Fazla boşlukları düzelt
    text = re.sub(r'\s+', ' ', text)

    # 4. Tarih, başlık vb. yapıları temizle (gerekirse)
    text = re.sub(r'(No:\s*\d+\s*-\s*\d+)', '', text)
    text = re.sub(r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}', '', text)
    text = re.sub(r'DECISION OF THE MONETARY POLICY COMMITTEE', '', text, flags=re.IGNORECASE)

    # 5. Baştaki boşlukları kes
    return text.strip()


In [ ]:
folder_path = "/content/gdrive/MyDrive/LLM/CBLLMMODEL/Dataset/DecisionReports"

# PDF dosyalarını listele
pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".pdf")]

In [ ]:
# pdf_files = ["Summary_2019_3_April.pdf"]


clean_sentence_list = []
Decision_Report_list = []

for pdf_name in pdf_files:
    pdf_path = f"Dataset/DecisionReports/{pdf_name}"

    pdf_text = extract_txt_from_pdf(pdf_path)
    doc = nlp(pdf_text)
    print(pdf_name,"-------")

    for sentence in doc.sents:

        words = sentence.split()
        corrected_words = [spell.correction(word) for word in words]
        corrected_sentence = " ".join(corrected_words)


        print(sentence)
        clean_sentence_list.append(sentence.text)
        Decision_Report_list.append(pdf_name)
        break
    break





In [ ]:
len(clean_sentence_list)

In [ ]:
import pandas as pd
Sentence_Df = pd.DataFrame(columns = ["Sentence","Report", "Label",])
Sentence_Df["Sentence"] = clean_sentence_list
Sentence_Df["Report"] = Decision_Report_list
def clean_illegal_chars(text):
    if isinstance(text, str):
        return re.sub(r'[\x00-\x1F\x7F-\x9F\u2028\u2029]', '', text)
    return text

Sentence_Df = Sentence_Df.applymap(clean_illegal_chars)
Sentence_Df.to_excel("Decision_Sentence_Df.xlsx")

In [ ]:
Sentence_Df

In [ ]:
Sentence_Df.loc[0, "Sentence"]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("vennify/t5-base-grammar-correction")
model = AutoModelForSeq2SeqLM.from_pretrained("vennify/t5-base-grammar-correction")

text = "A  revision in the monetary policy stance may be considered, should the fiscal stance deviate significantly from this framework and consequently have an advers e effect on the medium-term inflation outlook. "
input_text = "grammar: " + text
inputs = tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True)
outputs = model.generate(inputs, max_length=128)
print(outputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
import pandas as pd
Df = pd.read_excel("Filtered_Df.xlsx", index_col = "index" )

In [ ]:
Df.head()

In [ ]:
for row in Df.index:
    text = Df.loc[row, "Sentence"]
    Df.loc[row, "Label"] = len(text)

In [ ]:
df_sorted = Df.sort_values(by='Label')

In [ ]:
df_sorted.head()

In [ ]:
df_sorted.to_excel("Df_Filtered_Sorted.xlsx")

In [ ]:
i = 1
for row in Df.index:

    text = Df.loc[row, "Sentence"]
    input_text = "grammar: " + text
    inputs = tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(inputs, max_length=128)
    res = tokenizer.decode(outputs[0], skip_special_tokens=True)

    text = Df.loc[row, "Sentence"] = res
    print(i, text)
    i+=1


In [ ]:
from textblob import TextBlob

#function to convert string to list
def convert(lst):
    return ([i for item in lst for i in item.split()])

#add your string instead yor stng
lst =  ['yor stng']
#here we convert string to list using the function
lst = convert(lst)
#initislising the mistakes variable
mistakes = 0
#printing the list to show if text is correct
print(lst)
#here we take each item from list and correct it if it was not equal to original text that means that it has a mistake and if it is equal to old word then it does not have mistake
for x in lst:
  a = TextBlob(x)
  if (a.correct() != x):
    mistakes = mistakes + 1

#printing the number of mistakes
print(mistakes)

In [ ]:
0